# Evaluating Survey Area for various survey modes, in baseline survey v5.3 #

- creation date : 2026-10-07
- notebook on survey stragegy : https://github.com/lsst/survey-strategy_lsst_io/blob/main/notebooks/SurveyAreas.ipynb
- conda install -c conda-forge tabulate

## Overview and outputs

This notebook measures the **sky area** and the **number of visits** of each survey mode (WFD, mini-surveys, DDF, twilight, ToO...) in the baseline simulation, using the scheduler target map and MAF.

Steps:
1. **Target map labels**: show the dust map and the label map (which pixel belongs to which survey mode), and compute the area of each label.
2. **Micro-surveys**: count visits and plot visit maps for all visits, DDF, RGES, twilight near-sun and triplet third visits.
3. **Visits needed to cover each region**: attribute visits to regions with `WFDlabelStacker` and plot one visit map per region.
4. **Visits per point in each region**: per-band visit maps over the whole sky, then median/mean inside each region.

**Input**: the OpSim database `baseline_v5.3.6_10yrs.db` (path set in the `opsdb` variable).

**Outputs**: MAF `.npz` files go to `DATA02_SurveyAreas/`. The figures of this notebook are only displayed (`b.plot()` does not save them); `FIG02_SurveyAreas/` is created for figures saved later.

## 0. Setup

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import colorcet as cc
import pandas as pd
import healpy as hp
from rubin_scheduler.scheduler.utils import CurrentAreaMap

# Output directories: MAF .npz files go to DATA_DIR, all figures go to FIG_DIR
DATA_DIR = "DATA02_SurveyAreas"
FIG_DIR = "FIG02_SurveyAreas"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

## 1. Target map and region labels

`CurrentAreaMap` returns the scheduler goal weights per filter (`maps`) and one survey-mode label per HEALPix pixel (`labels`). DDFs and other micro-surveys are added later by the scheduler, so they are not in these maps.

In [ ]:
nside = 64

# get_current_footprint returns the 'target map' weightings over the sky per filter,
# and the 'labels' indicating the primary survey purpose in that area
# note that DDF and other microsurveys get added in the FBS, so are not included in these maps

skymap = CurrentAreaMap(nside=nside)  # , dust_limit=0.12)
maps, labels = skymap.return_maps()

### Figure 1 - Dust map and WFD footprint

Mollweide map of the dust map used by the footprint (`skymap.dustmap`, colour scale 0-1.2). The transparency encodes the footprint: WFD pixels are opaque (alpha 1), other pixels are more transparent (0.7), and pixels that are either dusty (dust >= 0.12) or not observed in r are made 0.3 more transparent. A graticule is overplotted.

In [ ]:
wfd = np.where(
    (labels == "lowdust")
    | (labels == "euclid_overlap")
    | (labels == "bulgy")
    | (labels == "virgo")
    | (labels == "LMC_SMC"),
    1,
    0,
)
alpha = np.where(wfd == 1, 1, 0.7)
alpha = np.where((skymap.dustmap < 0.12) & (maps["r"] > 0), alpha, alpha - 0.3)
hp.mollview(skymap.dustmap, min=0, max=1.2, alpha=alpha, title=None, cbar=False)
hp.graticule()

**Label list.** `label_names` lists the survey-mode labels present in the map (empty label excluded). `label_dict` gives each label an integer id, used by the next figure.

In [ ]:
# what labels do we have?
label_names = [l for l in np.unique(labels) if l != ""]
label_names

In [ ]:
# let's show them on the sky

label_dict = {}
for i, name in enumerate(label_names):
    label_dict[name] = i + 1
label_dict

### Figure 2 - Label map

Sky map where each pixel is coloured by the integer id of its survey-mode label (categorical `glasbey` colour map, graticule on, no colour bar). Unlabelled pixels are NaN (blank). The id-to-name correspondence is given by `label_dict` above.

In [ ]:
ft = np.zeros(len(maps["r"]))
for name in label_names:
    match = np.where(labels == name)
    ft[match] += label_dict[name]
ft = np.round(ft).astype(int)
ft = np.where(ft == 0, np.nan, ft)

glasbey_cmap = ListedColormap(cc.glasbey_category10, name="glasbey_category10")

# Pass the colormap object to projview
_ = hp.newvisufunc.projview(ft, cmap=glasbey_cmap, graticule=True, cbar=False)

**Area of each label (table).** Area = number of pixels of the label times the pixel area (deg2). `Approx NPointing` converts that area into a number of telescope pointings: one field of view covers about 9.6 deg2, and a factor 1.3 accounts for the overlap between pointings and for visits that hang over the footprint edge. This is only an approximate estimate.

In [ ]:
# Area on-sky in each of these regions?
pixarea = hp.nside2pixarea(nside, degrees=True)
# fudge factor to cover X amount of area
# overlap between pointings in the tesselation is 10%,
# but edge effects of the footprint increases the needed
# number of visits (to fully cover the area, as some visits 'hang over' the edge)
# NOTE THIS IS APPROXIMATE and will vary with footprint details
pixarea_to_pointings = (1 / 9.6) * 1.3

skarea = {}
skapprox_pointings = {}
for name in label_names:
    npix = len(np.where(labels == name)[0])
    skarea[name] = npix * pixarea
    skapprox_pointings[name] = np.ceil(skarea[name] * pixarea_to_pointings)
    # this would be 1.1 for a 10% overlap between pointings, but
# allfootprint
npix = len(np.where(labels != "")[0])
skarea["All"] = npix * pixarea
skapprox_pointings["All"] = np.ceil(skarea["All"] * pixarea_to_pointings)

print("Area per SKY_AREA section")
pd.DataFrame([skarea, skapprox_pointings], index=["Area (sq deg)", "Approx NPointing"]).round(1).T

## 2. Visits of the baseline simulation

Load the visits (time, band, pointing, exposure time, `scheduler_note`) from the OpSim database into a DataFrame.

In [ ]:
# how many visits in the current baseline fall into each of these regions?
from rubin_sim.data import get_baseline
import sqlite3
from rubin_sim.maf import WFDlabelStacker
import rubin_sim.maf as maf

# opsdb = get_baseline()
# opsdb = "baseline_v5.3.6_10yrs.db"
opsdb = "/Users/dagoret/DATA/OpSim/baseline_v5.3.6_10yrs.db"
print(opsdb)
conn = sqlite3.connect(opsdb)
opsim = os.path.split(opsdb)[-1].replace(".db", "")
print(opsim, ":", opsdb)
visits = pd.read_sql(
    "select observationStartMJD, band, fieldRA, fieldDec, rotSkyPos, visitExposureTime, target_name, scheduler_note from observations",
    conn,
)

visits.head()

### Micro-surveys: visit counts

Visits of the micro-surveys are selected directly from `scheduler_note`: near-sun twilight (`TwilightNEO`), DDF (`DD`, without RGES), `RGES`, and the third visit of triplets. `nvis` stores the number of visits of each.

In [ ]:
# Microsurveys - count up total visits by labelling directly
twilight_neo = visits.query("scheduler_note.str.contains('twilight_near_sun')")
ddf = visits.query("scheduler_note.str.contains('DD') and not scheduler_note.str.contains('RGES')")
rges = visits.query("scheduler_note.str.contains('RGES')")
triplets = visits.query("scheduler_note.str.contains('long') and not scheduler_note.str.contains('blob')")
nvis = {}
nvis["all visits"] = len(visits)
nvis["TwilightNEO"] = len(twilight_neo)
nvis["DD"] = len(ddf)
nvis["Triplet (third)"] = len(triplets)
nvis["RGES"] = len(rges)
nvis

### Figures 3-7 - Visit maps of the micro-surveys

For each selection, in this order: `all visits` (all except ToO), `DD`, `RGES`, `TwilightNEO` and `Triplet (third)`, the cell:
- counts visits per HEALPix pixel (`CountMetric`, nside 64), and plots the map with the colour scale clipped at the 95th percentile;
- prints the summary statistics (mean, median, rms of the visit count over the sky);
- stores the covered area (pixels with at least one visit), the approximate number of pointings and the median/mean number of visits, for the final table.

These figures are displayed only, not saved.

In [ ]:
area = {}
approx_pointings = {}
med_nvis = {}
mean_nvis = {}

plot_dict = {"percentile_clip": 95}

name = "all visits"
print(f"Plotting for {name}")
vis = visits.query("not scheduler_note.str.contains('ToO')").to_records()
if len(vis) > 0:
    metric = maf.CountMetric(col="observationStartMJD")
    slicer = maf.HealpixSlicer(nside=nside)
    summaries = [maf.MeanMetric(), maf.MedianMetric(), maf.RmsMetric()]
    b = maf.MetricBundle(
        metric,
        slicer,
        constraint="",
        summary_metrics=summaries,
        run_name=opsim,
        info_label=name,
        plot_dict=plot_dict,
    )
    g = maf.MetricBundleGroup({0: b}, None, out_dir=DATA_DIR)
    g.run_current(sim_data=vis, constraint="")

    area[name] = len(np.where(b.metric_values.filled(0) > 0)[0]) * pixarea
    approx_pointings[name] = np.ceil(area[name] * pixarea_to_pointings)
    med_nvis[name] = b.summary_values["Median"]
    mean_nvis[name] = b.summary_values["Mean"]

    print(b.summary_values)
    figs = b.plot()

name = "DD"
print(f"Plotting for {name}")
vis = ddf.to_records()
if len(vis) > 0:
    metric = maf.CountMetric(col="observationStartMJD")
    slicer = maf.HealpixSlicer(nside=nside)
    summaries = [maf.MeanMetric(), maf.MedianMetric(), maf.RmsMetric()]
    b = maf.MetricBundle(
        metric,
        slicer,
        constraint="",
        summary_metrics=summaries,
        run_name=opsim,
        info_label=name,
        plot_dict=plot_dict,
    )
    g = maf.MetricBundleGroup({0: b}, None, out_dir=DATA_DIR)
    g.run_current(sim_data=vis, constraint="")

    area[name] = len(np.where(b.metric_values.filled(0) > 0)[0]) * pixarea
    approx_pointings[name] = np.ceil(area[name] * pixarea_to_pointings)
    med_nvis[name] = b.summary_values["Median"]
    mean_nvis[name] = b.summary_values["Mean"]

    print(b.summary_values)
    figs = b.plot()

name = "RGES"
print(f"Plotting for {name}")
vis = rges.to_records()
if len(vis) > 0:
    metric = maf.CountMetric(col="observationStartMJD")
    slicer = maf.HealpixSlicer(nside=nside)
    summaries = [maf.MeanMetric(), maf.MedianMetric(), maf.RmsMetric()]
    b = maf.MetricBundle(
        metric,
        slicer,
        constraint="",
        summary_metrics=summaries,
        run_name=opsim,
        info_label=name,
        plot_dict=plot_dict,
    )
    g = maf.MetricBundleGroup({0: b}, None, out_dir=DATA_DIR)
    g.run_current(sim_data=vis, constraint="")

    area[name] = len(np.where(b.metric_values.filled(0) > 0)[0]) * pixarea
    approx_pointings[name] = np.ceil(area[name] * pixarea_to_pointings)
    med_nvis[name] = b.summary_values["Median"]
    mean_nvis[name] = b.summary_values["Mean"]

    print(b.summary_values)
    figs = b.plot()

name = "TwilightNEO"
print(f"Plotting for {name}")
vis = twilight_neo.to_records()
if len(vis) > 0:
    metric = maf.CountMetric(col="observationStartMJD")
    slicer = maf.HealpixSlicer(nside=nside)
    summaries = [maf.MeanMetric(), maf.MedianMetric(), maf.RmsMetric()]
    b = maf.MetricBundle(
        metric,
        slicer,
        constraint="",
        summary_metrics=summaries,
        run_name=opsim,
        info_label=name,
        plot_dict=plot_dict,
    )
    g = maf.MetricBundleGroup({0: b}, None, out_dir=DATA_DIR)
    g.run_current(sim_data=vis, constraint="")

    area[name] = len(np.where(b.metric_values.filled(0) > 0)[0]) * pixarea
    approx_pointings[name] = np.ceil(area[name] * pixarea_to_pointings)
    med_nvis[name] = b.summary_values["Median"]
    mean_nvis[name] = b.summary_values["Mean"]

    print(b.summary_values)
    figs = b.plot()

name = "Triplet (third)"
print(f"Plotting for {name}")
vis = triplets.to_records()
if len(vis) > 0:
    metric = maf.CountMetric(col="observationStartMJD")
    slicer = maf.HealpixSlicer(nside=nside)
    summaries = [maf.MeanMetric(), maf.MedianMetric(), maf.RmsMetric()]
    b = maf.MetricBundle(
        metric,
        slicer,
        constraint="",
        summary_metrics=summaries,
        run_name=opsim,
        info_label=name,
        plot_dict=plot_dict,
    )
    g = maf.MetricBundleGroup({0: b}, None, out_dir=DATA_DIR)
    g.run_current(sim_data=vis, constraint="")

    area[name] = len(np.where(b.metric_values.filled(0) > 0)[0]) * pixarea
    approx_pointings[name] = np.ceil(area[name] * pixarea_to_pointings)
    med_nvis[name] = b.summary_values["Median"]
    mean_nvis[name] = b.summary_values["Mean"]

    print(b.summary_values)
    figs = b.plot()

## 3. Visits needed to cover each region

`other_visits` are all visits except DDF, near-sun twilight and ToO. A visit can count towards several regions, so `WFDlabelStacker` tags each visit with an `area_id`: a visit is attributed to a region if more than 40% of its field of view falls inside that region's pixels. For each region the next cells then count the attributed visits (`nvis`) and plot their map.

All the maps below show the number of visits per pixel (default MAF colour scale), are displayed only and are not saved.

In [ ]:
other_visits = visits.query(
    "~scheduler_note.str.contains('DD') and ~scheduler_note.str.contains('twilight_near_sun') and ~scheduler_note.str.contains('ToO')"
)
ovisitsnp = other_visits.to_records()

In [ ]:
footpts = {}

### Figure 8 - NES (North Ecliptic Spur)

Region: label `nes`.

In [ ]:
# Visits can count towards more than one area, but there will be a minimum number of visits required to cover a given area
# The WFDlabelStacker can "attribute" a visit towards a given area --
# the threshold is if >0.4 of the visit area falls into the "footpt" healpix than it is counted toward covering that area.

name = "NES"
footpts[name] = np.where(labels == "nes", 1, 0)
stacker = WFDlabelStacker(footprint=footpts[name], area_id_name=name)
ovisitsnp = stacker.run(sim_data=ovisitsnp)
nvis[name] = len(np.where(ovisitsnp["area_id"] == name)[0])

print(f"Plotting for {name}")
vis = ovisitsnp[np.where(ovisitsnp["area_id"] == name)]
metric = maf.CountMetric(col="observationStartMJD")
slicer = maf.HealpixSlicer(nside=nside)
summaries = [maf.MeanMetric(), maf.MedianMetric(), maf.RmsMetric()]
b = maf.MetricBundle(
    metric, slicer, constraint="", summary_metrics=summaries, run_name=opsim, info_label=name
)
g = maf.MetricBundleGroup({0: b}, None, out_dir=DATA_DIR)
g.run_current(sim_data=vis, constraint="")

area[name] = len(np.where(b.metric_values.filled(0) > 0)[0]) * pixarea
approx_pointings[name] = np.ceil(area[name] * pixarea_to_pointings)
med_nvis[name] = b.summary_values["Median"]
mean_nvis[name] = b.summary_values["Mean"]

print(b.summary_values)
figs = b.plot()

### Figure 9 - SCP (South Celestial Pole)

Region: label `scp`.

In [ ]:
name = "SCP"
footpts[name] = np.where(labels == "scp", 1, 0)
stacker = WFDlabelStacker(footprint=footpts[name], area_id_name=name)
ovisitsnp = stacker.run(sim_data=ovisitsnp)
nvis[name] = len(np.where(ovisitsnp["area_id"] == name)[0])

print(f"Plotting for {name}")
vis = ovisitsnp[np.where(ovisitsnp["area_id"] == name)]
metric = maf.CountMetric(col="observationStartMJD")
slicer = maf.HealpixSlicer(nside=nside)
summaries = [maf.MeanMetric(), maf.MedianMetric(), maf.RmsMetric()]
b = maf.MetricBundle(
    metric, slicer, constraint="", summary_metrics=summaries, run_name=opsim, info_label=name
)
g = maf.MetricBundleGroup({0: b}, None, out_dir=DATA_DIR)
g.run_current(sim_data=vis, constraint="")

area[name] = len(np.where(b.metric_values.filled(0) > 0)[0]) * pixarea
approx_pointings[name] = np.ceil(area[name] * pixarea_to_pointings)
med_nvis[name] = b.summary_values["Median"]
mean_nvis[name] = b.summary_values["Mean"]

print(b.summary_values)
figs = b.plot()

### Figure 10 - MW (Milky Way)

Region: labels `bulgy`, `dusty_plane` and `LMC_SMC` together.

In [ ]:
name = "MW"
footpts[name] = np.where((labels == "bulgy") | (labels == "dusty_plane") | (labels == "LMC_SMC"), 1, 0)
stacker = WFDlabelStacker(footprint=footpts[name], area_id_name=name)
ovisitsnp = stacker.run(sim_data=ovisitsnp)
nvis[name] = len(np.where(ovisitsnp["area_id"] == name)[0])

print(f"Plotting for {name}")
vis = ovisitsnp[np.where(ovisitsnp["area_id"] == name)]
metric = maf.CountMetric(col="observationStartMJD")
slicer = maf.HealpixSlicer(nside=nside)
summaries = [maf.MeanMetric(), maf.MedianMetric(), maf.RmsMetric()]
b = maf.MetricBundle(
    metric, slicer, constraint="", summary_metrics=summaries, run_name=opsim, info_label=name
)
g = maf.MetricBundleGroup({0: b}, None, out_dir=DATA_DIR)
g.run_current(sim_data=vis, constraint="")

area[name] = len(np.where(b.metric_values.filled(0) > 0)[0]) * pixarea
approx_pointings[name] = np.ceil(area[name] * pixarea_to_pointings)
med_nvis[name] = b.summary_values["Median"]
mean_nvis[name] = b.summary_values["Mean"]

print(b.summary_values)
figs = b.plot()

### Figure 11 - MW non-WFD

Region: label `dusty_plane` only, i.e. the Milky Way region that is not part of the WFD.

In [ ]:
name = "MW non-WFD"
footpts[name] = np.where((labels == "dusty_plane"), 1, 0)
stacker = WFDlabelStacker(footprint=footpts[name], area_id_name=name)
ovisitsnp = stacker.run(sim_data=ovisitsnp)
nvis[name] = len(np.where(ovisitsnp["area_id"] == name)[0])

print(f"Plotting for {name}")
vis = ovisitsnp[np.where(ovisitsnp["area_id"] == name)]
metric = maf.CountMetric(col="observationStartMJD")
slicer = maf.HealpixSlicer(nside=nside)
summaries = [maf.MeanMetric(), maf.MedianMetric(), maf.RmsMetric()]
b = maf.MetricBundle(
    metric, slicer, constraint="", summary_metrics=summaries, run_name=opsim, info_label=name
)
g = maf.MetricBundleGroup({0: b}, None, out_dir=DATA_DIR)
g.run_current(sim_data=vis, constraint="")

area[name] = len(np.where(b.metric_values.filled(0) > 0)[0]) * pixarea
approx_pointings[name] = np.ceil(area[name] * pixarea_to_pointings)
med_nvis[name] = b.summary_values["Median"]
mean_nvis[name] = b.summary_values["Mean"]

print(b.summary_values)
figs = b.plot()

### Figure 12 - ExGal (extragalactic WFD)

Region: labels `lowdust` and `euclid_overlap`.

In [ ]:
name = "ExGal"
footpts[name] = np.where((labels == "lowdust") | (labels == "euclid_overlap"), 1, 0)
stacker = WFDlabelStacker(footprint=footpts[name], area_id_name=name)
ovisitsnp = stacker.run(sim_data=ovisitsnp)
nvis[name] = len(np.where(ovisitsnp["area_id"] == name)[0])

print(f"Plotting for {name}")
vis = ovisitsnp[np.where(ovisitsnp["area_id"] == name)]
metric = maf.CountMetric(col="observationStartMJD")
slicer = maf.HealpixSlicer(nside=nside)
summaries = [maf.MeanMetric(), maf.MedianMetric(), maf.RmsMetric()]
b = maf.MetricBundle(
    metric, slicer, constraint="", summary_metrics=summaries, run_name=opsim, info_label=name
)
g = maf.MetricBundleGroup({0: b}, None, out_dir=DATA_DIR)
g.run_current(sim_data=vis, constraint="")

area[name] = len(np.where(b.metric_values.filled(0) > 0)[0]) * pixarea
approx_pointings[name] = np.ceil(area[name] * pixarea_to_pointings)
med_nvis[name] = b.summary_values["Median"]
mean_nvis[name] = b.summary_values["Mean"]

print(b.summary_values)
figs = b.plot()

### Figure 13 - Dust < 0.12

Region: `lowdust` pixels with dust map value below 0.12 (the dust cut of the footprint).

In [ ]:
name = "Dust<0.12"
footpts[name] = np.where((skymap.dustmap < 0.12) & (labels == "lowdust"), 1, 0)
stacker = WFDlabelStacker(footprint=footpts[name], area_id_name=name)
ovisitsnp = stacker.run(sim_data=ovisitsnp)
nvis[name] = len(np.where(ovisitsnp["area_id"] == name)[0])

print(f"Plotting for {name}")
vis = ovisitsnp[np.where(ovisitsnp["area_id"] == name)]
metric = maf.CountMetric(col="observationStartMJD")
slicer = maf.HealpixSlicer(nside=nside)
summaries = [maf.MeanMetric(), maf.MedianMetric(), maf.RmsMetric()]
b = maf.MetricBundle(
    metric, slicer, constraint="", summary_metrics=summaries, run_name=opsim, info_label=name
)
g = maf.MetricBundleGroup({0: b}, None, out_dir=DATA_DIR)
g.run_current(sim_data=vis, constraint="")

area[name] = len(np.where(b.metric_values.filled(0) > 0)[0]) * pixarea
approx_pointings[name] = np.ceil(area[name] * pixarea_to_pointings)
med_nvis[name] = b.summary_values["Median"]
mean_nvis[name] = b.summary_values["Mean"]

print(b.summary_values)
figs = b.plot()

### Figure 14 - WFD (all WFD regions)

Region: labels `lowdust`, `euclid_overlap`, `bulgy`, `virgo` and `LMC_SMC` together.

In [ ]:
name = "WFD"
footpts[name] = np.where(
    (labels == "lowdust")
    | (labels == "euclid_overlap")
    | (labels == "bulgy")
    | (labels == "virgo")
    | (labels == "LMC_SMC"),
    1,
    0,
)
stacker = WFDlabelStacker(footprint=footpts[name], area_id_name=name)
ovisitsnp = stacker.run(sim_data=ovisitsnp)
nvis[name] = len(np.where(ovisitsnp["area_id"] == name)[0])

print(f"Plotting for {name}")
vis = ovisitsnp[np.where(ovisitsnp["area_id"] == name)]
metric = maf.CountMetric(col="observationStartMJD")
slicer = maf.HealpixSlicer(nside=nside)
summaries = [maf.MeanMetric(), maf.MedianMetric(), maf.RmsMetric()]
b = maf.MetricBundle(
    metric, slicer, constraint="", summary_metrics=summaries, run_name=opsim, info_label=name
)
g = maf.MetricBundleGroup({0: b}, None, out_dir=DATA_DIR)
g.run_current(sim_data=vis, constraint="")

area[name] = len(np.where(b.metric_values.filled(0) > 0)[0]) * pixarea
approx_pointings[name] = np.ceil(area[name] * pixarea_to_pointings)
med_nvis[name] = b.summary_values["Median"]
mean_nvis[name] = b.summary_values["Mean"]

print(b.summary_values)
figs = b.plot()

### Summary table: visits and area per survey mode

Columns: `Nvisits` (number of visits), `Percent` (share of all visits; twilight is divided by 2 because its visits are half as long), `Area` (deg2 with at least one visit), `Approx NPointings`, and the median and mean number of visits per pixel. ToO visits are added as one more row.

In [ ]:
# Totalling up based on *visits required to cover a given region (or for a particular microsurvey)*
fraction = dict(zip(list(nvis.keys()), np.array(list(nvis.values())) / nvis["all visits"] * 100))
fraction["TwilightNEO"] /= 2  # visits are half as long

nvis["ToO"] = len(visits.query("scheduler_note.str.contains('ToO')"))
fraction["ToO"] = nvis["ToO"] / nvis["all visits"] * 100
pd.DataFrame(
    [nvis, fraction, area, approx_pointings, med_nvis, mean_nvis],
    index=["Nvisits", "Percent", "Area", "Approx NPointings", "Median Nvis", "Mean Nvis"],
).round(1).T

## 4. Visits per point within each region

A simpler estimate of the number of visits per sky position: count visits per pixel over the whole sky, separately for each band (`u g r i z y`) and for all bands, then take the pixels of each region from the maps (`footpts`). No figure is drawn here; the results are the three tables below (median, mean, and both merged).

In [ ]:
# The methods above were the only way to count up the total number of visits which would be required to completely cover each region.
# However, we can get an easier estimate of the number of visits per point *within* each region by just calculating over the whole sky
# And then subsetting the resulting healpix map.

other_visits = visits.query(
    "~scheduler_note.str.contains('DD') and ~scheduler_note.str.contains('twilight_near_sun') and ~scheduler_note.str.contains('ToO')"
)

metric = maf.CountMetric(col="observationStartMJD")
slicer = maf.HealpixSlicer(nside=nside)
summaries = [maf.MeanMetric(), maf.MedianMetric(), maf.RmsMetric()]

bundles = {}
for band in "ugrizy":
    vis = other_visits.query("band == @band").to_records()
    bundles[band] = maf.MetricBundle(
        metric, slicer, constraint=f"{band} band", summary_metrics=summaries, run_name=opsim, info_label=name
    )
    g = maf.MetricBundleGroup({band: bundles[band]}, None, out_dir=DATA_DIR)
    g.run_current(sim_data=vis, constraint=f"{band} band")

vis = other_visits.to_records()
bundles["all"] = maf.MetricBundle(
    metric, slicer, constraint="", summary_metrics=summaries, run_name=opsim, info_label=name
)
g = maf.MetricBundleGroup({"all": bundles["all"]}, None, out_dir=DATA_DIR)
g.run_current(sim_data=vis, constraint="")

**Per-region statistics.** For each region (`footpts`) and each band, the median and mean number of visits over the region's pixels. Tables: `med <band>` (median), `mean <band>` (mean), then both merged in one table.

In [ ]:
median_stats = {}
mean_stats = {}
for name in footpts:
    mean_stats[name] = {}
    median_stats[name] = {}
    mask = np.where(footpts[name] > 0)[0]
    for band in "ugrizy":
        mval = bundles[band].metric_values.filled(0)[mask]
        median_stats[name][band] = np.median(mval)
        mean_stats[name][band] = np.mean(mval)
    mval = bundles["all"].metric_values.filled(0)[mask]
    median_stats[name]["all"] = np.median(mval)
    mean_stats[name]["all"] = np.mean(mval)

In [ ]:
dd = pd.DataFrame(median_stats).T.astype(int)
dd.columns = [f"med {b}" for b in dd.columns]
dd

In [ ]:
dd2 = pd.DataFrame(mean_stats).T.astype(int)
dd2.columns = [f"mean {b}" for b in dd2.columns]
dd2

In [ ]:
dd.merge(dd2, right_index=True, left_index=True)